In [ ]:
from DenoiseST import DenoiseST
import torch
import pandas as pd
import scanpy as sc

In [ ]:
n_clusters = 4
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
filePath = './GateDiffST/CID4465'
h5ad = 'human_bc_spatial_CID4465.h5ad'
adata = sc.read_h5ad(filePath + '/' + h5ad)
adata

AnnData object with n_obs × n_vars = 1211 × 15128
    obs: 'nCount_RNA', 'nFeature_RNA', 'subtype', 'patientid', 'Classification'
    var: 'n_cells'
    uns: 'spatial', 'spatial_neighbors'
    obsm: 'spatial'
    obsp: 'spatial_connectivities', 'spatial_distances'

In [ ]:
# define model
model = DenoiseST(adata,device=device,n_top_genes=4096)
# train model
adata = model.train()

Begin to train ST data...


100%|██████████| 600/600 [00:15<00:00, 39.53it/s]


Optimization finished for ST data!


In [ ]:
from repair_model import main_repair
df=pd.DataFrame(adata.obsm['emb'])
csv_file = "example.csv"
main_repair(adata,df,device,save_path=csv_file)
data_df = pd.read_csv(csv_file, header=None)
data_df = data_df.values
adata.obsm['emb'] = data_df
adata

/root/miniconda3/envs/DiffusionST/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing: 100%|██████████| 1211/1211 [1:03:48<00:00,  3.16s/it]


[[ 1.09191871  0.11130592  0.34083721 ...  0.05238561  0.12884147
  -0.00888815]
 [ 0.07423603  0.20153664  0.22501834 ...  0.35305503  0.12824427
  -0.00248607]
 [ 0.07908592  0.13887832 -0.09049092 ...  0.07346712  0.12795959
  -0.06394646]
 ...
 [ 0.48334211  0.16935003  0.23408274 ...  0.28537574  0.23249622
   0.22905888]
 [ 0.50184089  0.15170594 -0.02416051 ... -0.00647782  0.14542985
   0.15827139]
 [-0.02351447  0.04210453 -0.01462429 ...  0.27469242  0.24605961
   0.22618809]]
0 : [[ 1.09191871  0.11130592  0.34083721 ...  0.05238561  0.12884147
  -0.00888815]
 [ 0.07423603  0.20153664  0.22501834 ...  0.35305503  0.12824427
  -0.00248607]
 [ 0.07908592  0.13887832 -0.09049092 ...  0.07346712  0.12795959
  -0.06394646]
 ...
 [ 0.48334211  0.16935003  0.23408274 ...  0.28537574  0.23249622
   0.22905888]
 [ 0.50184089  0.15170594 -0.02416051 ... -0.00647782  0.14542985
   0.15827139]
 [-0.02351447  0.04210453 -0.01462429 ...  0.27469242  0.24605961
   0.22618809]]
[[ 1.2321890

AnnData object with n_obs × n_vars = 1211 × 15128
    obs: 'nCount_RNA', 'nFeature_RNA', 'subtype', 'patientid', 'Classification'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'mean', 'std'
    uns: 'spatial', 'spatial_neighbors', 'hvg', 'log1p'
    obsm: 'spatial', 'distance_matrix', 'graph_neigh', 'adj', 'feat', 'feat_a', 'emb'
    obsp: 'spatial_connectivities', 'spatial_distances'

In [6]:
# set radius to specify the number of neighbors considered during refinement
radius = 50

tool = 'mclust' # mclust, leiden, and louvain

# clustering
from utils import clustering

if tool == 'mclust':
   clustering(adata, n_clusters, radius=radius, method=tool, refinement=True) # For DLPFC dataset, we use optional refinement step.
elif tool in ['leiden', 'louvain']:
   clustering(adata, n_clusters, radius=radius, method=tool, start=0.1, end=2.0, increment=0.01, refinement=False)

R[write to console]:                    __           __ 
   ____ ___  _____/ /_  _______/ /_
  / __ `__ \/ ___/ / / / / ___/ __/
 / / / / / / /__/ / /_/ (__  ) /_  
/_/ /_/ /_/\___/_/\__,_/____/\__/   version 6.1.1
Type 'citation("mclust")' for citing this R package in publications.



fitting ...
  |======================================================================| 100%


In [ ]:
plots = sc.pl.spatial(adata,
            img_key="lowres",
            color=["Classification", "domain"],
            title=["ground_truth","GateDiffST"],
            show=True)
